<a href="https://colab.research.google.com/github/LOVELY1907/SAP_AI-ML-DL/blob/main/03_Deep_Learning/IMAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import VGG16

In [ ]:
file_path = '/content/archive.zip'
extract_path = '/content'

In [ ]:
with zipfile.ZipFile(file_path,'r') as zip_ref:
  zip_ref.extractall(extract_path)

In [ ]:
#training and testing directory
training_directory = os.path.join(extract_path,"train")
testing_directory = os.path.join(extract_path,"test")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
val_generator = train_datagen.flow_from_directory(
    testing_directory,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 28 images belonging to 2 classes.


In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers:
  layer.trainable = False

In [ ]:
from tensorflow.keras import Model
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
optimizer='adam',
loss='binary_crossentropy',
metrics=['accuracy']
)

In [ ]:
callbacks = [
EarlyStopping(patience=5, restore_best_weights=True),
ModelCheckpoint("dog_cat_cnn.h5", save_best_only=True)
]

In [ ]:
history = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.5743 - loss: 1.7300 

14/14 ━━━━━━━━━━━━━━━━━━━━ 257s 18s/step - accuracy: 0.5774 - loss: 1.6968 - val_accuracy: 0.9286 - val_loss: 0.3168
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 252s 18s/step - accuracy: 0.8139 - loss: 0.4449 - val_accuracy: 0.7857 - val_loss: 0.3803
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 251s 18s/step - accuracy: 0.8530 - loss: 0.3516 - val_accuracy: 0.8214 - val_loss: 0.3362
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 251s 18s/step - accuracy: 0.9047 - loss: 0.2758 - val_accuracy: 0.8214 - val_loss: 0.3329
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 255s 18s/step - accuracy: 0.9090 - loss: 0.2192 - val_accuracy: 0.8571 - val_loss: 0.3956
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.9089 - loss: 0.2121 

14/14 ━━━━━━━━━━━━━━━━━━━━ 249s 18s/step - accuracy: 0.9087 - loss: 0.2122 - val_accuracy: 0.8571 - val_loss: 0.3063
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.9015 - loss: 0.2045 

14/14 ━━━━━━━━━━━━━━━━━━━━ 256s 18s/step - accuracy: 0.9025 - loss: 0.2031 - val_accuracy: 0.7857 - val_loss: 0.3031
Epoch 8/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 248s 18s/step - accuracy: 0.9187 - loss: 0.2006 - val_accuracy: 0.8571 - val_loss: 0.3182
Epoch 9/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.9219 - loss: 0.1843 

14/14 ━━━━━━━━━━━━━━━━━━━━ 256s 18s/step - accuracy: 0.9223 - loss: 0.1842 - val_accuracy: 0.9286 - val_loss: 0.2492
Epoch 10/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 248s 18s/step - accuracy: 0.9253 - loss: 0.1885 - val_accuracy: 0.7857 - val_loss: 0.5393


In [ ]:
test_generator = test_datagen.flow_from_directory(
    testing_directory,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
)
test_loss, test_accuracy = model.evaluate(test_generator)
print("Test Accuracy:", test_accuracy)

Found 140 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


5/5 ━━━━━━━━━━━━━━━━━━━━ 75s 14s/step - accuracy: 0.7821 - loss: 0.5342
Test Accuracy: 0.7928571701049805


In [ ]:
model=Sequential([
    Conv2D(32,(3,3),activation='relu',
           input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("dog_cat_cnn.h5", save_best_only=True)
]

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    training_directory,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    testing_directory,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 447 images belonging to 2 classes.
Found 28 images belonging to 2 classes.


In [ ]:
from IPython.core import history
history = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 58s 4s/step - accuracy: 0.6814 - loss: 0.5643 - val_accuracy: 0.7143 - val_loss: 0.5512
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.6907 - loss: 0.5872 - val_accuracy: 0.7143 - val_loss: 0.5558
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.6681 - loss: 0.5933 - val_accuracy: 0.7143 - val_loss: 0.5362
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.7350 - loss: 0.5622 - val_accuracy: 0.7143 - val_loss: 0.5411
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.7221 - loss: 0.5612 - val_accuracy: 0.7857 - val_loss: 0.5170
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 58s 4s/step - accuracy: 0.6920 - loss: 0.5668 - val_accuracy: 0.7500 - val_loss: 0.5316
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7325 - loss: 0.5244

14/14 ━━━━━━━━━━━━━━━━━━━━ 51s 4s/step - accuracy: 0.7334 - loss: 0.5247 - val_accuracy: 0.7500 - val_loss: 0.4612
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7581 - loss: 0.5290

14/14 ━━━━━━━━━━━━━━━━━━━━ 51s 4s/step - accuracy: 0.7579 - loss: 0.5289 - val_accuracy: 0.7857 - val_loss: 0.4288
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.7304 - loss: 0.5140 - val_accuracy: 0.7500 - val_loss: 0.5447
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.6693 - loss: 0.5703 - val_accuracy: 0.7143 - val_loss: 0.4838
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.6874 - loss: 0.5445 - val_accuracy: 0.7857 - val_loss: 0.4658
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.7353 - loss: 0.5034 - val_accuracy: 0.7857 - val_loss: 0.4892
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.7374 - loss: 0.5080 - val_accuracy: 0.7857 - val_loss: 0.4831


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model('/content/dog_cat_cnn.h5')

img_path = '/content/dogs/dog_114.jpg'
img = image.load_img(img_path, target_size=IMG_SIZE)
img = image.img_to_array(img) / 255.0
img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

print("Dog" if prediction[0][0] > 0.5 else "Cat")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
Dog


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model('/content/dog_cat_cnn.h5')

img_path = '/content/dogs/dog_534.jpg'
img = image.load_img(img_path, target_size=IMG_SIZE)
img = image.img_to_array(img) / 255.0
img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

print("Dog" if prediction[0][0] > 0.5 else "Cat")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
Cat
